# Performance & Benchmarking Tests

**Release:** v1.0.0 | **Date:** 2026-03-17

## Success Criteria: SC-01 Performance Parity
**Target**: Read/write latency within 10-15% of native tables

## Test Matrix
| Test | Approach | Success Criteria |
|------|----------|------------------|
| Large-Scale Scans | Full scans of 1M+ rows | Latency within 10% of native |
| Point Lookups | Search Optimization enabled | Sub-second response |
| Complex Joins | Multi-way joins | Execution plan efficiency |
| Aggregations | GROUP BY, window functions | Performance parity |
| Clustering | Auto-clustering depth/overlap | Convergence verification |
| **Managed vs Customer Storage** | Compare Iceberg storage backends | Within 5% of each other |

## Table Types Under Test (4-Way Comparison)
| Schema | Storage Type | DDL Syntax | Description |
|--------|--------------|------------|-------------|
| NATIVE_BASELINE | Snowflake Native | `CREATE TABLE` | Standard Snowflake tables (baseline) |
| MANAGED_ICEBERG | Managed Storage | `CATALOG='SNOWFLAKE' ICEBERG_VERSION=3` | Iceberg v3 with Snowflake-managed storage |
| TESTS | Customer Storage | `CATALOG=SNOWFLAKE EXTERNAL_VOLUME=EXVOL` | Iceberg v3 with customer External Volume |
| EXTERNAL_ICEBERG | Customer + Path | `+ BASE_LOCATION='path/'` | Iceberg v3 with explicit paths for interop |

In [ ]:
-- Set context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICEBERG_POC;
USE WAREHOUSE COMPUTE_WH;

---
## Test 1: Large-Scale Scan - 4-Way Comparison
Compare Native, Managed Storage, Customer Storage, and External Path Iceberg tables.

In [ ]:
-- Verify all baseline healthcare data exists (4 storage types)
SELECT '1-Native'            AS storage_type, 'NATIVE_BASELINE.ENCOUNTERS' AS table_name, COUNT(*) AS row_count FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS
UNION ALL
SELECT '2-Managed',           'MANAGED_ICEBERG.ENCOUNTERS',  COUNT(*) FROM ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS
UNION ALL
SELECT '3-Customer',          'TESTS.ENCOUNTERS',            COUNT(*) FROM ICEBERG_POC.TESTS.ENCOUNTERS
UNION ALL
SELECT '4-External',          'EXTERNAL_ICEBERG.ENCOUNTERS', COUNT(*) FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
ORDER BY storage_type;

In [ ]:
-- Native table full scan: encounters by type
SELECT
    encounter_type,
    COUNT(*)                       AS encounter_count,
    COUNT(DISTINCT patient_id)     AS unique_patients,
    ROUND(AVG(total_charge), 2)    AS avg_charge,
    ROUND(SUM(total_charge), 2)    AS total_revenue
FROM ICEBERG_POC.NATIVE_BASELINE.ENCOUNTERS
GROUP BY encounter_type
ORDER BY encounter_count DESC;

In [ ]:
-- Internal Iceberg table full scan: encounters by type
SELECT
    encounter_type,
    COUNT(*)                       AS encounter_count,
    COUNT(DISTINCT patient_id)     AS unique_patients,
    ROUND(AVG(total_charge), 2)    AS avg_charge,
    ROUND(SUM(total_charge), 2)    AS total_revenue
FROM ICEBERG_POC.TESTS.ENCOUNTERS
GROUP BY encounter_type
ORDER BY encounter_count DESC;

In [ ]:
-- Managed Storage Iceberg table full scan: encounters by type
SELECT
    encounter_type,
    COUNT(*)                       AS encounter_count,
    COUNT(DISTINCT patient_id)     AS unique_patients,
    ROUND(AVG(total_charge), 2)    AS avg_charge,
    ROUND(SUM(total_charge), 2)    AS total_revenue
FROM ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS
GROUP BY encounter_type
ORDER BY encounter_count DESC;

In [ ]:
-- External Iceberg table full scan: encounters by type
SELECT
    encounter_type,
    COUNT(*)                       AS encounter_count,
    COUNT(DISTINCT patient_id)     AS unique_patients,
    ROUND(AVG(total_charge), 2)    AS avg_charge,
    ROUND(SUM(total_charge), 2)    AS total_revenue
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
GROUP BY encounter_type
ORDER BY encounter_count DESC;

---
## Test 2: External Iceberg - Multi-Table Scan
Test all external Iceberg tables.

In [ ]:
-- External PATIENTS scan: patient distribution by state and insurance plan
SELECT
    state,
    insurance_plan,
    COUNT(*)        AS patient_count,
    COUNT(DISTINCT city) AS cities_covered
FROM ICEBERG_POC.EXTERNAL_ICEBERG.PATIENTS
GROUP BY state, insurance_plan
ORDER BY patient_count DESC
LIMIT 20;

In [ ]:
-- External CLAIMS scan: payer performance metrics
SELECT
    payer_name,
    claim_status,
    COUNT(*)                              AS claim_count,
    ROUND(AVG(billed_amount), 2)          AS avg_billed,
    ROUND(SUM(billed_amount), 2)          AS total_billed,
    ROUND(SUM(paid_amount), 2)            AS total_paid,
    ROUND(AVG(paid_amount / NULLIF(billed_amount, 0)) * 100, 1) AS avg_payment_pct
FROM ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS
GROUP BY payer_name, claim_status
ORDER BY total_billed DESC;

In [ ]:
-- External MEDICATIONS scan with VARIANT: medication form and refill distribution
SELECT
    drug_name,
    medication_details:form::VARCHAR          AS medication_form,
    COUNT(*)                                  AS prescription_count,
    SUM(CASE WHEN medication_details:controlled::BOOLEAN = TRUE THEN 1 ELSE 0 END) AS controlled_count,
    ROUND(AVG(medication_details:refills::INT), 1) AS avg_refills,
    COUNT(DISTINCT patient_id)                AS unique_patients
FROM ICEBERG_POC.EXTERNAL_ICEBERG.MEDICATIONS
GROUP BY drug_name, medication_details:form::VARCHAR
ORDER BY prescription_count DESC
LIMIT 20;

---
## Test 3: Point Lookups with Search Optimization

In [ ]:
-- Enable Search Optimization on Internal Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.ENCOUNTERS
  ADD SEARCH OPTIMIZATION ON EQUALITY(patient_id, encounter_id);

In [ ]:
-- Enable Search Optimization on Managed Storage Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS
  ADD SEARCH OPTIMIZATION ON EQUALITY(patient_id, encounter_id);

In [ ]:
-- Enable Search Optimization on External Iceberg table
ALTER ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
  ADD SEARCH OPTIMIZATION ON EQUALITY(patient_id, encounter_id);

In [ ]:
-- Point lookup: Internal Iceberg — patient encounters
SELECT * FROM ICEBERG_POC.TESTS.ENCOUNTERS WHERE patient_id = 1001;

In [ ]:
-- Point lookup: External Iceberg — patient encounters
SELECT * FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS WHERE patient_id = 1001;

In [ ]:
-- Point lookup: Managed Storage Iceberg — patient encounters
SELECT * FROM ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS WHERE patient_id = 1001;

---
## Test 4: Auto-Clustering Comparison

In [ ]:
-- Enable clustering on all Iceberg tables by encounter_type and encounter_date for range scans
ALTER ICEBERG TABLE ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS CLUSTER BY (encounter_type, encounter_date);
ALTER ICEBERG TABLE ICEBERG_POC.TESTS.ENCOUNTERS CLUSTER BY (encounter_type, encounter_date);
ALTER ICEBERG TABLE ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS CLUSTER BY (encounter_type, encounter_date);

In [ ]:
-- Check clustering info for ENCOUNTERS across all Iceberg storage types
SELECT 'Managed'  AS storage_type, SYSTEM$CLUSTERING_INFORMATION('ICEBERG_POC.MANAGED_ICEBERG.ENCOUNTERS') AS clustering_info
UNION ALL
SELECT 'Customer', SYSTEM$CLUSTERING_INFORMATION('ICEBERG_POC.TESTS.ENCOUNTERS')
UNION ALL
SELECT 'External', SYSTEM$CLUSTERING_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS');

---
## Test 5: Complex Joins - External Tables

In [ ]:
-- Multi-way join on External Iceberg tables: provider specialty utilization by state
SELECT
    p.state,
    pr.specialty,
    COUNT(DISTINCT e.encounter_id)     AS encounter_count,
    COUNT(DISTINCT e.patient_id)       AS unique_patients,
    ROUND(SUM(e.total_charge), 2)      AS total_revenue,
    ROUND(AVG(e.total_charge), 2)      AS avg_charge
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS e
JOIN ICEBERG_POC.EXTERNAL_ICEBERG.PATIENTS   p  ON e.patient_id  = p.patient_id
JOIN ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS  pr ON e.provider_id = pr.provider_id
GROUP BY p.state, pr.specialty
ORDER BY total_revenue DESC
LIMIT 20;

In [ ]:
-- Join: payer claims performance by provider specialty
SELECT
    pr.specialty,
    c.payer_name,
    c.claim_status,
    COUNT(*)                               AS claim_count,
    ROUND(SUM(c.billed_amount), 2)         AS total_billed,
    ROUND(SUM(c.paid_amount), 2)           AS total_paid,
    ROUND(AVG(c.allowed_amount / NULLIF(c.billed_amount, 0)) * 100, 1) AS avg_allowed_pct
FROM ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS      c
JOIN ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS  e  ON c.encounter_id = e.encounter_id
JOIN ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS   pr ON e.provider_id  = pr.provider_id
GROUP BY pr.specialty, c.payer_name, c.claim_status
ORDER BY total_billed DESC
LIMIT 20;

---
## Test 6: VARIANT Query Performance - External Tables

In [ ]:
-- Query VARIANT in External CLAIMS: range scan by submitted_date
SELECT
    payer_name,
    claim_status,
    COUNT(*)                       AS claims_in_range,
    ROUND(SUM(billed_amount), 2)   AS total_billed
FROM ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS
WHERE submitted_date BETWEEN DATEADD(day, -90, CURRENT_DATE()) AND CURRENT_DATE()
GROUP BY payer_name, claim_status
ORDER BY total_billed DESC;

In [ ]:
-- Query VARIANT in External MEDICATIONS: medication details metadata
SELECT
    drug_name,
    medication_details:form::VARCHAR        AS form,
    medication_details:controlled::BOOLEAN  AS is_controlled,
    COUNT(*)                                AS rx_count,
    ROUND(AVG(medication_details:refills::INT), 1) AS avg_refills,
    COUNT(DISTINCT patient_id)              AS unique_patients
FROM ICEBERG_POC.EXTERNAL_ICEBERG.MEDICATIONS
GROUP BY drug_name, medication_details:form::VARCHAR, medication_details:controlled::BOOLEAN
ORDER BY rx_count DESC;

In [ ]:
-- Query VARIANT in External PROVIDERS: provider attributes metadata
SELECT
    specialty,
    provider_attributes:telehealth::BOOLEAN   AS offers_telehealth,
    COUNT(*)                                   AS provider_count,
    SUM(CASE WHEN accepting_patients THEN 1 ELSE 0 END) AS accepting_count,
    ARRAY_SIZE(provider_attributes:languages)  AS language_count
FROM ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS
GROUP BY specialty, provider_attributes:telehealth::BOOLEAN, ARRAY_SIZE(provider_attributes:languages)
ORDER BY provider_count DESC;

---
## Test 7: Aggregations & Window Functions

In [ ]:
-- Window functions on External ENCOUNTERS: running charge and visit sequence per patient
SELECT
    patient_id,
    encounter_type,
    encounter_date,
    total_charge,
    SUM(total_charge)    OVER (PARTITION BY patient_id ORDER BY encounter_date) AS running_total_charge,
    ROW_NUMBER()         OVER (PARTITION BY patient_id ORDER BY encounter_date) AS visit_sequence,
    LAG(total_charge)    OVER (PARTITION BY patient_id ORDER BY encounter_date) AS prev_charge
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
WHERE patient_id IN (1001, 1002, 1003)
ORDER BY patient_id, encounter_date
LIMIT 50;

---
## Test 8: Query Performance Analysis - All Table Types

In [ ]:
-- Compare performance across all 4 storage types for healthcare queries
SELECT
    CASE
        WHEN QUERY_TEXT ILIKE '%NATIVE_BASELINE%'   THEN '1-Native'
        WHEN QUERY_TEXT ILIKE '%MANAGED_ICEBERG%'   THEN '2-Managed'
        WHEN QUERY_TEXT ILIKE '%EXTERNAL_ICEBERG%'  THEN '4-External'
        WHEN QUERY_TEXT ILIKE '%TESTS.%'            THEN '3-Customer'
        ELSE 'Other'
    END AS storage_type,
    COUNT(*)                                                                     AS query_count,
    ROUND(AVG(TOTAL_ELAPSED_TIME) / 1000, 3)                                    AS avg_elapsed_sec,
    ROUND(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000, 3) AS p50_sec,
    ROUND(PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY TOTAL_ELAPSED_TIME) / 1000, 3) AS p95_sec,
    ROUND(AVG(BYTES_SCANNED) / 1024 / 1024, 2)                                  AS avg_mb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY_BY_SESSION())
WHERE QUERY_TYPE = 'SELECT'
    AND EXECUTION_STATUS = 'SUCCESS'
    AND (QUERY_TEXT ILIKE '%NATIVE_BASELINE%'
         OR QUERY_TEXT ILIKE '%MANAGED_ICEBERG%'
         OR QUERY_TEXT ILIKE '%EXTERNAL_ICEBERG%'
         OR QUERY_TEXT ILIKE '%TESTS.ENCOUNTERS%'
         OR QUERY_TEXT ILIKE '%TESTS.CLAIMS%'
         OR QUERY_TEXT ILIKE '%TESTS.PATIENTS%')
GROUP BY 1
ORDER BY 1;

---
## Test 9: Warehouse Size Scaling

In [ ]:
-- XS Warehouse - External Iceberg
USE WAREHOUSE POC_WH_XS;

SELECT 'XS - External' AS test, encounter_type, COUNT(*), ROUND(SUM(total_charge), 2), ROUND(AVG(total_charge), 2)
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
WHERE encounter_type = 'Emergency'
GROUP BY encounter_type;

In [ ]:
-- Medium Warehouse - External Iceberg
USE WAREHOUSE POC_WH_M;

SELECT 'M - External' AS test, encounter_type, COUNT(*), ROUND(SUM(total_charge), 2), ROUND(AVG(total_charge), 2)
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
WHERE encounter_type = 'Emergency'
GROUP BY encounter_type;

In [ ]:
-- Large Warehouse - External Iceberg
USE WAREHOUSE POC_WH_L;

SELECT 'L - External' AS test, encounter_type, COUNT(*), ROUND(SUM(total_charge), 2), ROUND(AVG(total_charge), 2)
FROM ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS
WHERE encounter_type = 'Emergency'
GROUP BY encounter_type;

In [ ]:
-- Reset warehouse
USE WAREHOUSE COMPUTE_WH;

---
## Test 10: External Storage Metadata Paths

In [ ]:
-- Verify external storage paths for Spark/Databricks interoperability
SELECT
    'PATIENTS'    AS table_name,
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.PATIENTS')):metadataLocation::STRING    AS metadata_path
UNION ALL
SELECT 'ENCOUNTERS',
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.ENCOUNTERS')):metadataLocation::STRING
UNION ALL
SELECT 'CLAIMS',
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.CLAIMS')):metadataLocation::STRING
UNION ALL
SELECT 'MEDICATIONS',
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.MEDICATIONS')):metadataLocation::STRING
UNION ALL
SELECT 'PROVIDERS',
    PARSE_JSON(SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ICEBERG_POC.EXTERNAL_ICEBERG.PROVIDERS')):metadataLocation::STRING;

---
## Summary: Performance Results

### 4-Way Storage Comparison
| Test | Native | Managed | Customer | External | Status |
|------|--------|---------|----------|----------|--------|
| Full Scan (1M rows) | TBD | TBD | TBD | TBD | ⏳ |
| Point Lookup | TBD | TBD | TBD | TBD | ⏳ |
| Multi-Join | TBD | TBD | TBD | TBD | ⏳ |
| VARIANT Query | N/A | TBD | TBD | TBD | ⏳ |
| Clustering Efficiency | N/A | TBD | TBD | TBD | ⏳ |

### Storage Type Reference
| Schema | Storage Type | DDL Syntax |
|--------|--------------|------------|
| NATIVE_BASELINE | Snowflake Native | `CREATE TABLE` |
| MANAGED_ICEBERG | Snowflake Managed | `CATALOG='SNOWFLAKE' ICEBERG_VERSION=3` |
| TESTS | Customer External Volume | `CATALOG=SNOWFLAKE EXTERNAL_VOLUME=EXVOL` |
| EXTERNAL_ICEBERG | Customer + Explicit Path | `+ BASE_LOCATION='path/'` |

### Healthcare Tables (per storage type)
| Table | Rows | Purpose |
|-------|------|---------|
| ENCOUNTERS | 1M | Main benchmark - visit records |
| PATIENTS | 100K | PHI governance testing |
| CLAIMS | 500K | DML operations |
| MEDICATIONS | 300K | VARIANT column support |
| PROVIDERS | 1K | Reference/dimension table |

*Run query history analysis cells to populate actual performance metrics*